# ApprovalMax Great Expectations Validation

Validates the Gold CDC lifecycle mart and writes results to `workspace.engineer_support_monitoring.great_expectations_results`.


In [ ]:
from datetime import datetime, timezone
import json

from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, BooleanType

catalog = 'workspace'
gold_schema = 'engineer_support_gold'
monitoring_schema = 'engineer_support_monitoring'

dbt_gold_table = f'{catalog}.{gold_schema}.fact_approval_document_lifecycle_dbt'
notebook_gold_table = f'{catalog}.{gold_schema}.fact_approval_document_lifecycle'
result_table = f'{catalog}.{monitoring_schema}.great_expectations_results'

spark.sql(f'USE CATALOG {catalog}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{monitoring_schema}')

def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name).limit(1).count()
        return True
    except Exception:
        return False

gold_table = dbt_gold_table if table_exists(dbt_gold_table) else notebook_gold_table
validation_run_id = f'gx_approvalmax_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}'

result_schema = StructType([
    StructField('validation_run_id', StringType(), False),
    StructField('table_name', StringType(), False),
    StructField('expectation_name', StringType(), False),
    StructField('status', StringType(), False),
    StructField('success', BooleanType(), False),
    StructField('failed_row_count', LongType(), True),
    StructField('severity', StringType(), False),
    StructField('checked_at', TimestampType(), False),
    StructField('details', StringType(), True),
])

spark.sql(f'''CREATE TABLE IF NOT EXISTS {result_table} (
  validation_run_id STRING,
  table_name STRING,
  expectation_name STRING,
  status STRING,
  success BOOLEAN,
  failed_row_count BIGINT,
  severity STRING,
  checked_at TIMESTAMP,
  details STRING
) USING DELTA''')

print(f'Validating: {gold_table}')
print(f'Results: {result_table}')


In [ ]:
expectations = [
    {'name': 'expect_document_id_not_null', 'severity': 'critical', 'sql': f'SELECT * FROM {gold_table} WHERE document_id IS NULL'},
    {'name': 'expect_company_id_not_null', 'severity': 'critical', 'sql': f'SELECT * FROM {gold_table} WHERE company_id IS NULL'},
    {'name': 'expect_document_status_in_allowed_values', 'severity': 'critical', 'sql': f"SELECT * FROM {gold_table} WHERE document_status NOT IN ('draft', 'submitted', 'approved', 'rejected', 'paid', 'cancelled') OR document_status IS NULL"},
    {'name': 'expect_total_amount_non_negative', 'severity': 'critical', 'sql': f'SELECT * FROM {gold_table} WHERE total_amount < 0'},
    {'name': 'expect_approval_cycle_time_non_negative', 'severity': 'critical', 'sql': f'SELECT * FROM {gold_table} WHERE approval_cycle_time_minutes < 0'},
    {'name': 'expect_gold_grain_unique_by_document_id', 'severity': 'critical', 'sql': f'SELECT document_id, COUNT(*) AS row_count FROM {gold_table} GROUP BY document_id HAVING COUNT(*) > 1'},
    {'name': 'expect_required_currency_not_null', 'severity': 'warning', 'sql': f'SELECT * FROM {gold_table} WHERE currency IS NULL'},
]

checked_at = datetime.now(timezone.utc)
results = []

for exp in expectations:
    failed_df = spark.sql(exp['sql'])
    failed_count = failed_df.count()
    success = failed_count == 0
    status = 'SUCCESS' if success else 'FAILED'
    print(f"{exp['name']}: {status}, failed_row_count={failed_count}")
    results.append({
        'validation_run_id': validation_run_id,
        'table_name': gold_table,
        'expectation_name': exp['name'],
        'status': status,
        'success': bool(success),
        'failed_row_count': int(failed_count),
        'severity': exp['severity'],
        'checked_at': checked_at,
        'details': json.dumps({'sql': exp['sql']}),
    })

result_df = spark.createDataFrame(results, schema=result_schema)
result_df.write.format('delta').mode('append').saveAsTable(result_table)
display(result_df)

critical_failures = [r for r in results if (not r['success']) and r['severity'] == 'critical']
if critical_failures:
    raise Exception('Great Expectations validation failed: ' + json.dumps(critical_failures, default=str))

print('All critical Great Expectations validations passed')
